# Dissertation diagnostic review notebook

This reporting-only derivative of `06_final_observable_factorial_deep_hedging_H035_ANNUAL_COMPLETE_SUPPORT_FINAL_DIAGNOSTICS` regenerates candidate dissertation diagnostics from the locked exported results. **It does not train or refit any model.**

The notebook has two purposes:

1. produce updated, presentation-ready versions of diagnostics compatible with the final H035 observable-only design;
2. display historically reported but superseded diagnostics in a separate section so they can be approved, rejected or retained only as future-research material.

All figures are saved under `updated_figures/`, and a machine-readable review manifest is written at the end.

In [1]:

from __future__ import annotations

import json
import math
import os
import shutil
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display, Markdown
from PIL import Image as PILImage, ImageDraw

DATA_DIR = Path(os.environ.get("HEDGE_REVIEW_DATA_DIR", "/mnt/data/diagnostic_review/full")).resolve()
FIGURE_DIR = Path(os.environ.get("HEDGE_REVIEW_FIGURE_DIR", "/mnt/data/diagnostic_review/updated_figures")).resolve()
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OLD_MEDIA_DIR = Path("/mnt/data/diagnostic_review/old_media")
SUPERSEDED_DIR = Path("/mnt/data/diagnostic_review/superseded_outputs")

PRIMARY = "lifted_heston__pooled__sim_only"
MARKET = "market_only__pooled"
BS = "black_scholes_delta"
VENDOR = "vendor_delta"

LABELS = {
    PRIMARY: "Lifted Heston pooled, simulation-only",
    "lifted_heston__pooled__head_only": "Lifted Heston pooled, head-only",
    "lifted_heston__pooled__full": "Lifted Heston pooled, full adaptation",
    MARKET: "Market-only pooled",
    BS: "Black-Scholes delta",
    VENDOR: "Vendor delta",
    "heston__partial__full": "Heston partial, full adaptation",
    "lifted_heston__partial__full": "Lifted Heston partial, full adaptation",
    "black_scholes__partial__full": "Black-Scholes partial, full adaptation",
    "heston_delta": "Heston analytical delta",
    "lifted_heston_delta": "Lifted-Heston analytical delta",
    "no_hedge": "No hedge",
}

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "legend.fontsize": 8.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TABLES = {}
def load(name, **kwargs):
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path, **kwargs)
    TABLES[name] = frame
    return frame

headline = load("headline_summary.csv")
headline_1m = load("headline_1m_matched.csv")
paired_dollar = load("paired_dollar_summary.csv")
paired_dollar_seed = load("paired_dollar_seed_summary.csv")
regime_paired = load("paired_win_rate_by_regime.csv")
seed_headline = load("neural_seed_headline.csv")
cost_sensitivity = load("cost_sensitivity.csv")
win_maturity_cost = load("paired_win_rate_by_maturity_cost.csv")
training_history = load("training_history.csv")
quote_funnel = load("quote_funnel.csv")
coverage = load("coverage_by_year_bucket.csv")
no_trade = load("no_trade_selection.csv")
no_trade_boundary = load("no_trade_boundary_audit.csv")
walk_forward = load("walk_forward_family.csv", parse_dates=["evaluation_start"])
concurrent_load = load("concurrent_capital_load.csv", parse_dates=["date"])
cumulative_paths = load("cumulative_paired_dollar_paths.csv", parse_dates=["exit_date"])
fold_support = load("fold_support.csv")
cohort_exclusions = load("cohort_exclusions.csv")
collateral = load("account_collateral_sensitivity.csv")

result_cols = [
    "episode_id", "model_cell", "underlying_cost_bps", "training_seed", "fold",
    "entry_maturity_bucket", "normalised_pnl", "net_pnl_usd", "squared_error",
    "turnover", "option_pnl_points", "stock_gain_points", "financing_pnl",
    "dividend_carry", "underlying_cost_points", "option_spread_cost_points",
    "fixed_option_fee_points", "net_pnl_points",
]
episode_results = load("episode_results.csv", usecols=result_cols)

signature = json.loads((DATA_DIR / "experiment_result_signature.json").read_text())
assert len(episode_results) == 719_370, len(episode_results)
assert episode_results["fold"].nunique() == 9
assert set(episode_results["underlying_cost_bps"].unique()) == {1.0, 5.0, 10.0}

REVIEW_ROWS = []

def save_show(fig, filename, *, title, category, placement, importance, caveats, recommendation):
    path = FIGURE_DIR / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    REVIEW_ROWS.append({
        "title": title,
        "category": category,
        "filename": filename,
        "proposed_placement": placement,
        "why_important": importance,
        "caveats": caveats,
        "preliminary_recommendation": recommendation,
    })
    display(Image(filename=str(path)))
    return path


def heatmap(ax, frame, *, title, fmt=".2f", center=None, cmap="coolwarm", cbar_label=None):
    values = frame.to_numpy(dtype=float)
    if center is None:
        im = ax.imshow(values, aspect="auto", cmap=cmap)
    else:
        bound = np.nanmax(np.abs(values - center))
        bound = max(float(bound), 1e-9)
        im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=center-bound, vmax=center+bound)
    ax.set_xticks(np.arange(frame.shape[1]))
    ax.set_xticklabels(frame.columns)
    ax.set_yticks(np.arange(frame.shape[0]))
    ax.set_yticklabels(frame.index)
    ax.set_title(title)
    for i in range(frame.shape[0]):
        for j in range(frame.shape[1]):
            v = values[i, j]
            if np.isfinite(v):
                ax.text(j, i, format(v, fmt), ha="center", va="center", fontsize=8)
    if cbar_label:
        cb=fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label(cbar_label)
    return im

print({
    "status": "LOCKED_RESULTS_LOADED",
    "rows": len(episode_results),
    "folds": episode_results.fold.nunique(),
    "signature": signature.get("result_signature", "")[:16],
    "training_performed": False,
    "output": str(FIGURE_DIR),
})


{'status': 'LOCKED_RESULTS_LOADED', 'rows': 719370, 'folds': 9, 'signature': '8b1a044129f69723', 'training_performed': False, 'output': '/mnt/data/diagnostic_review/updated_figures'}


## 1. Terminal-loss exceedance curves

In [2]:

models=[PRIMARY, MARKET, BS, VENDOR]
rows=episode_results.loc[
    episode_results.underlying_cost_bps.eq(1.0)
    & episode_results.model_cell.isin(models),
    ["episode_id","model_cell","normalised_pnl"]
].groupby(["model_cell","episode_id"],as_index=False).normalised_pnl.mean()
rows["adverse_loss"]=(-rows.normalised_pnl).clip(lower=0)
maximum=float(rows.adverse_loss.quantile(.995))
thresholds=np.linspace(0,maximum,110)
fig,ax=plt.subplots(figsize=(9.2,5.4))
for model in models:
    loss=rows.loc[rows.model_cell.eq(model),"adverse_loss"].to_numpy(float)
    curve=[np.mean(loss>t) for t in thresholds]
    ax.plot(thresholds,curve,label=LABELS[model],linewidth=2.2)
ax.set_title("Adverse terminal-loss exceedance at one basis point")
ax.set_xlabel("Premium-normalised adverse terminal loss threshold")
ax.set_ylabel("Probability of exceeding threshold")
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
ax.grid(alpha=.25)
save_show(fig,"01_terminal_loss_exceedance.png",
 title="Terminal-loss exceedance curves",category="Compatible with final design",
 placement="Main results, immediately after the MSE bias-variance and tail decomposition in RQ1.",
 importance="Shows whether lower MSE reflects a broad distributional shift or protection concentrated in the far adverse tail. It directly supports the claim that deep hedging transforms risk rather than generating alpha.",
 caveats="The neural curves average fitted seeds at the episode level. The chart is descriptive and does not create additional independent observations beyond the nine annual folds.",
 recommendation="APPROVE - main text")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/01_terminal_loss_exceedance.png')

## 2. Risk-turnover frontier

In [3]:

front=headline_1m.loc[headline_1m.underlying_cost_bps.eq(1.0)].copy()
front=front.sort_values(["turnover","premium_normalised_rmse"])
# Pareto frontier: retain points that improve RMSE as turnover increases.
best=np.inf; pareto=[]
for row in front.itertuples():
    if row.premium_normalised_rmse < best:
        pareto.append(row)
        best=row.premium_normalised_rmse
fig,ax=plt.subplots(figsize=(9.2,5.8))
baseline=front.model_type.eq("baseline")
ax.scatter(front.loc[baseline,"turnover"],front.loc[baseline,"premium_normalised_rmse"],alpha=.45,label="Analytical baselines",s=45)
ax.scatter(front.loc[~baseline,"turnover"],front.loc[~baseline,"premium_normalised_rmse"],alpha=.55,label="Neural policies",s=45)
if pareto:
    ax.plot([r.turnover for r in pareto],[r.premium_normalised_rmse for r in pareto],linestyle="--",linewidth=1.4,label="Observed lower envelope")
key_specs={
    PRIMARY:("Leading Lifted sim-only",(14,-18)),
    MARKET:("Market-only pooled",(12,14)),
    BS:("Black-Scholes delta",(-95,16)),
    VENDOR:("Vendor delta",(-75,-20)),
}
for model,(short_label,offset) in key_specs.items():
    g=front.loc[front.model_cell.eq(model)]
    if len(g):
        r=g.iloc[0]
        ax.scatter([r.turnover],[r.premium_normalised_rmse],s=80,edgecolor="black",linewidth=.7,zorder=5)
        ax.annotate(short_label,(r.turnover,r.premium_normalised_rmse),xytext=offset,textcoords="offset points",fontsize=8,
                    arrowprops=dict(arrowstyle="-",linewidth=.7),bbox=dict(boxstyle="round,pad=.2",fc="white",alpha=.85,ec="none"))
ax.set_title("Matched 1M hedging-risk and turnover frontier")
ax.set_xlabel("Mean stock turnover")
ax.set_ylabel("Premium-normalised RMSE")
ax.grid(alpha=.25)
ax.legend(frameon=False,loc="best")
save_show(fig,"02_risk_turnover_frontier.png",
 title="Risk-turnover frontier",category="Compatible with final design",
 placement="Main results, after the headline performance table or immediately before the RQ1 mechanism discussion.",
 importance="Recasts the model comparison as a control-efficiency problem. It reveals whether lower hedging error is obtained at a defensible intervention intensity or through disproportionately aggressive trading.",
 caveats="The plotted lower envelope is descriptive, not a formally estimated efficient frontier. The 1M matched-support restriction should be stated prominently.",
 recommendation="APPROVE - main text")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/02_risk_turnover_frontier.png')

## 3. Paired one-contract dollar P&L difference

In [4]:

models=[PRIMARY,"lifted_heston__pooled__head_only","lifted_heston__pooled__full",MARKET]
plot=paired_dollar.loc[paired_dollar.model_cell.isin(models) & paired_dollar.baseline_cell.isin([BS,VENDOR])].copy()
plot["label"]=plot.model_cell.map(LABELS)+" vs "+plot.baseline_cell.map(LABELS)
plot=plot.sort_values("fold_equal_mean_dollar_difference")
y=np.arange(len(plot))
mean=plot.fold_equal_mean_dollar_difference.to_numpy(float)
lo=plot.fold_block_ci_2_5pct.to_numpy(float); hi=plot.fold_block_ci_97_5pct.to_numpy(float)
fig,ax=plt.subplots(figsize=(9.4,6.0))
ax.errorbar(mean,y,xerr=np.vstack([mean-lo,hi-mean]),fmt="o",capsize=4)
ax.axvline(0,linewidth=1,color="black")
ax.set_yticks(y); ax.set_yticklabels(plot.label)
ax.set_xlabel("Fold-equal mean dollar P&L difference per SPX contract")
ax.set_title("Paired dollar P&L: learned policies minus delta benchmarks")
ax.grid(axis="x",alpha=.25)
save_show(fig,"03_paired_dollar_pnl_difference.png",
 title="Paired one-contract dollar P&L difference",category="Compatible with final design",
 placement="Main RQ1 results, directly after the terminal-loss or MSE-decomposition figure.",
 importance="Makes the central economic distinction tangible: the policies can reduce quadratic hedging risk while lowering average dollar P&L. This prevents the examiner from equating lower MSE with a more profitable short-volatility strategy.",
 caveats="Intervals resample annual folds. Dollar values represent one standard SPX contract and are not returns on funded capital.",
 recommendation="APPROVE - main text")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/03_paired_dollar_pnl_difference.png')

## 4. Causal regime-conditioned performance

In [5]:

reg=regime_paired.loc[regime_paired.model_cell.eq(PRIMARY) & regime_paired.baseline_cell.eq(BS)].copy()
reg=reg.loc[~reg.regime_dimension.eq("execution_gap_regime")]
reg=reg.groupby(["regime_dimension","regime"],as_index=False).agg(
    mse_difference=("mean_mse_difference","mean"),
    dollar_difference=("mean_dollar_difference","mean"),
    lower_error_rate=("probability_lower_squared_error","mean"),
    higher_pnl_rate=("probability_higher_dollar_pnl","mean"),
    matched_episodes=("matched_episodes","mean"),
)
pretty={"volatility_regime":"Volatility","skew_regime":"Skew","liquidity_regime":"Liquidity"}
reg["label"]=[f"{pretty.get(d,d)}: {r}" for d,r in zip(reg.regime_dimension,reg.regime)]
reg=reg.sort_values(["regime_dimension","regime"])
y=np.arange(len(reg))
fig,axes=plt.subplots(1,2,figsize=(11.5,5.2),sharey=True)
axes[0].barh(y,reg.mse_difference)
axes[0].axvline(0,color="black",linewidth=1)
axes[0].set_xlabel("Mean MSE difference (negative favours neural)")
axes[0].set_yticks(y); axes[0].set_yticklabels(reg.label)
axes[0].set_title("Quadratic-risk effect")
axes[1].barh(y,reg.dollar_difference)
axes[1].axvline(0,color="black",linewidth=1)
axes[1].set_xlabel("Mean dollar P&L difference")
axes[1].set_title("Dollar outcome")
for i,row in reg.reset_index(drop=True).iterrows():
    axes[0].text(row.mse_difference,i,f"  win {row.lower_error_rate:.0%}",va="center",fontsize=8)
    axes[1].text(row.dollar_difference,i,f"  P&L win {row.higher_pnl_rate:.0%}",va="center",fontsize=8)
fig.suptitle("Leading policy relative to Black-Scholes delta by causally defined market regime")
fig.tight_layout()
save_show(fig,"04_regime_conditioned_performance.png",
 title="Causal regime-conditioned performance",category="Compatible with final design",
 placement="Appendix robustness section, with one paragraph in the RQ1 discussion identifying the states in which the effect is concentrated.",
 importance="Moves the analysis from an unconditional average to an economic mechanism: it asks when the policy reduces risk and whether those same regimes improve or worsen dollar outcomes.",
 caveats="Regime thresholds are fold-specific and training-defined, but subgroup evidence is low-powered. Treat this as mechanism evidence, not a new confirmatory family.",
 recommendation="APPROVE - appendix; cite in main discussion")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/04_regime_conditioned_performance.png')

## 5. Accounting and P&L-component decomposition

In [6]:

models=[PRIMARY,MARKET,BS]
cols=["option_pnl_points","stock_gain_points","financing_pnl","dividend_carry","underlying_cost_points","option_spread_cost_points","fixed_option_fee_points","net_pnl_points"]
acc=episode_results.loc[episode_results.underlying_cost_bps.eq(1.0) & episode_results.model_cell.isin(models),["episode_id","model_cell"]+cols]
acc=acc.groupby(["model_cell","episode_id"],as_index=False)[cols].mean().groupby("model_cell")[cols].mean()
base=acc.loc[BS]
components=["stock_gain_points","financing_pnl","dividend_carry","underlying_cost_points","net_pnl_points"]
labels=["Stock hedge","Financing","Dividend carry","Underlying costs","Net P&L"]
fig,ax=plt.subplots(figsize=(9.3,5.5))
x=np.arange(len(components)); width=.34
for offset,model in [(-width/2,PRIMARY),(width/2,MARKET)]:
    vals=[]
    for c in components:
        diff=acc.loc[model,c]-base[c]
        if c=="underlying_cost_points": diff=-diff
        vals.append(diff)
    ax.bar(x+offset,vals,width,label=f"{LABELS[model]} minus BS delta")
ax.axhline(0,color="black",linewidth=1)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Mean difference in SPX index points")
ax.set_title("Where the paired P&L difference comes from")
ax.legend(frameon=False)
ax.grid(axis="y",alpha=.25)
save_show(fig,"05_accounting_decomposition.png",
 title="Accounting and P&L-component decomposition",category="Compatible with final design",
 placement="Appendix economic interpretation; summarise the dominant component in the RQ1 discussion.",
 importance="Connects learned hedge actions to the self-financing identity and reveals whether lower mean P&L is driven by stock trading, financing, carry or transaction costs.",
 caveats="Option P&L and option-entry costs are common across matched policies and are therefore omitted from the paired-difference bars. Underlying cost differences are shown as negative contributions.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/05_accounting_decomposition.png')

## 6. Optimisation-seed stability

In [7]:

models=[PRIMARY,"lifted_heston__pooled__head_only","lifted_heston__pooled__full",MARKET,"heston__partial__full"]
seed=seed_headline.loc[seed_headline.underlying_cost_bps.eq(1.0) & seed_headline.model_cell.isin(models)].copy()
order=models[::-1]
fig,ax=plt.subplots(figsize=(9.2,5.2))
for y,model in enumerate(order):
    g=seed.loc[seed.model_cell.eq(model)]
    ax.scatter(g.premium_normalised_mse,np.full(len(g),y),alpha=.75,s=45)
    ax.scatter([g.premium_normalised_mse.mean()],[y],marker="D",s=70,edgecolor="black",label="Mean" if y==0 else None)
ax.set_yticks(range(len(order))); ax.set_yticklabels([LABELS.get(m,m) for m in order])
ax.set_xlabel("Premium-normalised MSE")
ax.set_title("Optimisation-seed dispersion at one basis point")
ax.grid(axis="x",alpha=.25)
save_show(fig,"06_seed_stability.png",
 title="Optimisation-seed stability",category="Compatible with final design",
 placement="Appendix learning diagnostics; cite in RQ1 when discussing whether the leading cell is a lucky initialisation.",
 importance="Shows within-policy optimisation variance directly and allows the examiner to compare seed dispersion with between-policy effect sizes.",
 caveats="Five seeds are enough to expose gross instability but not to estimate the full optimisation distribution.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/06_seed_stability.png')

## 7. Transaction-cost sensitivity and tested break-even

In [8]:

cs=cost_sensitivity.loc[cost_sensitivity.model_cell.isin([PRIMARY,MARKET]) & cost_sensitivity.baseline_cell.eq(BS)].copy()
cs=cs.groupby(["model_cell","underlying_cost_bps","fold"],as_index=False).agg(
    mean_mse_difference=("mean_mse_difference","mean"),
    mean_dollar_difference=("mean_dollar_difference","mean"),
)
summary=cs.groupby(["model_cell","underlying_cost_bps"]).agg(
    mse_mean=("mean_mse_difference","mean"),mse_sem=("mean_mse_difference","sem"),
    dollar_mean=("mean_dollar_difference","mean"),dollar_sem=("mean_dollar_difference","sem"),
).reset_index()
fig,axes=plt.subplots(1,2,figsize=(11.2,4.8))
for model in [PRIMARY,MARKET]:
    g=summary.loc[summary.model_cell.eq(model)]
    axes[0].errorbar(g.underlying_cost_bps,g.mse_mean,yerr=1.96*g.mse_sem,marker="o",capsize=3,label=LABELS[model])
    axes[1].errorbar(g.underlying_cost_bps,g.dollar_mean,yerr=1.96*g.dollar_sem,marker="o",capsize=3,label=LABELS[model])
for ax in axes: ax.axhline(0,color="black",linewidth=1); ax.set_xlabel("Underlying transaction cost (bp)"); ax.grid(alpha=.25)
axes[0].set_ylabel("MSE difference vs BS delta")
axes[0].set_title("Quadratic-risk effect")
axes[1].set_ylabel("Dollar P&L difference per contract")
axes[1].set_title("Dollar outcome")
axes[1].legend(frameon=False)
fig.suptitle("Sensitivity to tested stock-transaction costs")
fig.tight_layout()
save_show(fig,"07_cost_sensitivity.png",
 title="Transaction-cost sensitivity and tested break-even",category="Compatible with final design",
 placement="Appendix robustness; cite in the daily-rehedging and practical-interpretation discussion.",
 importance="Tests whether the risk advantage survives plausible friction changes and whether a better hedge becomes an even worse trade as costs rise.",
 caveats="Only 1, 5 and 10 bp are observed. The figure must not extrapolate a break-even outside the tested range.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/07_cost_sensitivity.png')

## 8. Paired win rates by maturity and cost

In [9]:

win=win_maturity_cost.loc[win_maturity_cost.model_cell.eq(PRIMARY) & win_maturity_cost.baseline_cell.eq(BS)].copy()
win=win.groupby(["entry_maturity_bucket","underlying_cost_bps"],as_index=False).agg(
    lower_error=("probability_lower_squared_error","mean"),
    higher_pnl=("probability_higher_dollar_pnl","mean"),
)
order=["1W","2W","1M","2M","3M"]
a=win.pivot(index="entry_maturity_bucket",columns="underlying_cost_bps",values="lower_error").reindex(order)
b=win.pivot(index="entry_maturity_bucket",columns="underlying_cost_bps",values="higher_pnl").reindex(order)
fig,axes=plt.subplots(1,2,figsize=(10.8,4.8))
for ax,frame,title in [(axes[0],a,"Probability of lower squared error"),(axes[1],b,"Probability of higher dollar P&L")]:
    im=ax.imshow(frame.to_numpy(float),aspect="auto",vmin=0,vmax=1,cmap="viridis")
    ax.set_xticks(range(frame.shape[1])); ax.set_xticklabels([f"{c:g} bp" for c in frame.columns])
    ax.set_yticks(range(frame.shape[0])); ax.set_yticklabels(frame.index)
    ax.set_title(title)
    for i in range(frame.shape[0]):
        for j in range(frame.shape[1]): ax.text(j,i,f"{frame.iloc[i,j]:.0%}",ha="center",va="center",fontsize=8)
fig.colorbar(im,ax=axes.ravel().tolist(),label="Episode share",shrink=.85)
fig.suptitle("Leading policy versus Black-Scholes delta by maturity and cost")
fig.tight_layout()
save_show(fig,"08_win_rate_maturity_cost.png",
 title="Paired win rates by maturity and cost",category="Compatible with final design",
 placement="Appendix robustness, adjacent to the maturity-by-cost MSE heatmap.",
 importance="Shows whether average MSE gains arise from winning frequently or from preventing a small number of very large errors. The paired dollar win rate exposes the opposite objective.",
 caveats="Episode win rates are descriptive and dependent within annual folds. They must not be presented as binomial significance tests.",
 recommendation="APPROVE - appendix")


/tmp/ipykernel_2263/2060282938.py:19: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/08_win_rate_maturity_cost.png')

## 9. Training and validation histories

In [10]:

selectors=[
    ("Lifted pooled simulation", "lifted_heston","pooled","sim_only","simulation_selection"),
    ("Lifted pooled head-only", "lifted_heston","pooled","head_only","inner_selection"),
    ("Lifted pooled full", "lifted_heston","pooled","full","inner_selection"),
    ("Market-only pooled", "market_only","pooled","market_only","inner_selection"),
    ("Black-Scholes pooled full", "black_scholes","pooled","full","inner_selection"),
]
fig,axes=plt.subplots(2,3,figsize=(13,7.2))
for ax,(title,source,pooling,stage,scope) in zip(axes.flat,selectors):
    g=training_history.loc[
        training_history.source.eq(source)&training_history.pooling.eq(pooling)&
        training_history.checkpoint_stage.eq(stage)&training_history.fit_scope.eq(scope)
    ]
    agg=g.groupby("epoch").agg(train=("train_loss","median"),validation=("validation_loss","median")).reset_index()
    ax.plot(agg.epoch,agg.train,label="Train")
    ax.plot(agg.epoch,agg.validation,label="Validation")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(alpha=.25)
    if len(agg)>80: ax.set_xlim(1,80)
axes.flat[0].legend(frameon=False)
axes.flat[-1].axis("off")
fig.suptitle("Representative optimisation histories")
fig.tight_layout()
save_show(fig,"09_training_validation_curves.png",
 title="Training and validation histories",category="Compatible with final design",
 placement="Appendix learning diagnostics, before the saturation and no-op table.",
 importance="Distinguishes convergence, early stopping, negligible adaptation and Black-Scholes optimisation pathology. This supports the claim that nominal treatment labels do not guarantee realised learning.",
 caveats="Curves are medians across folds and seeds and therefore describe typical optimisation behaviour rather than one fitted checkpoint.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/09_training_validation_curves.png')

## 10. Quote funnel and maturity-by-year coverage

In [11]:

q=quote_funnel.copy()
cov=coverage.pivot(index="entry_year",columns="entry_maturity_bucket",values="episodes").reindex(columns=["1W","2W","1M","2M","3M"]).fillna(0)
fig,axes=plt.subplots(1,2,figsize=(13,5.2),gridspec_kw={"width_ratios":[1.05,1.35]})
axes[0].barh(np.arange(len(q)),q.rows)
axes[0].set_yticks(np.arange(len(q))); axes[0].set_yticklabels(q.stage.str.replace("_"," "))
axes[0].set_xscale("log"); axes[0].invert_yaxis(); axes[0].set_xlabel("Rows retained (log scale)"); axes[0].set_title("SPX quote funnel")
im=axes[1].imshow(cov.to_numpy(float),aspect="auto",cmap="Blues")
axes[1].set_xticks(range(cov.shape[1])); axes[1].set_xticklabels(cov.columns)
axes[1].set_yticks(range(cov.shape[0])); axes[1].set_yticklabels(cov.index.astype(int))
axes[1].set_title("Eligible market episodes by entry year and maturity")
for i in range(cov.shape[0]):
    for j in range(cov.shape[1]): axes[1].text(j,i,f"{cov.iloc[i,j]:.0f}",ha="center",va="center",fontsize=7)
fig.colorbar(im,ax=axes[1],label="Episodes",fraction=.046,pad=.04)
fig.tight_layout()
save_show(fig,"10_quote_funnel_and_coverage.png",
 title="Quote funnel and maturity-by-year coverage",category="Compatible with final design",
 placement="Data chapter after the contract-selection and support rules; alternatively Appendix A if the main text is crowded.",
 importance="Makes the empirical sample visible and explains why complete five-bucket support begins in 2016 rather than appearing as an arbitrary exclusion.",
 caveats="Raw quote-row counts and final episode counts are different units; the two panels should be interpreted separately.",
 recommendation="APPROVE - data chapter or Appendix A")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/10_quote_funnel_and_coverage.png')

## 11. Structural no-trade selection and boundary audit

In [12]:

med=no_trade_boundary.groupby(["baseline","cost_bps"]).selected_scale.median().unstack("cost_bps")
bound=no_trade_boundary.groupby(["baseline","cost_bps"]).selected_at_grid_boundary.mean().unstack("cost_bps")
fig,axes=plt.subplots(1,2,figsize=(10.8,4.5))
for ax,frame,title,fmt in [(axes[0],med,"Median selected scale",".1f"),(axes[1],bound,"Share selected at grid boundary",".0%")]:
    im=ax.imshow(frame.to_numpy(float),aspect="auto",cmap="viridis")
    ax.set_xticks(range(frame.shape[1])); ax.set_xticklabels([f"{c:g} bp" for c in frame.columns])
    ax.set_yticks(range(frame.shape[0])); ax.set_yticklabels(frame.index.str.replace("_"," "))
    ax.set_title(title)
    for i in range(frame.shape[0]):
        for j in range(frame.shape[1]): ax.text(j,i,format(frame.iloc[i,j],fmt),ha="center",va="center",fontsize=8)
fig.suptitle("Structural no-trade benchmark tuning")
fig.tight_layout()
save_show(fig,"11_no_trade_selection_boundary.png",
 title="Structural no-trade selection and boundary audit",category="Compatible with final design",
 placement="Appendix benchmark diagnostics.",
 importance="Shows that weak no-trade results may reflect thin inner-validation samples and boundary-constrained tuning rather than a rejection of no-trade theory.",
 caveats="A boundary selection is a warning about identification, not proof that a larger grid value would improve forward performance.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/11_no_trade_selection_boundary.png')

## 12. Walk-forward performance by policy family

In [13]:

wf=walk_forward.loc[walk_forward.policy_family.isin(["sim_only","market_only","black_scholes_delta","vendor_delta"])].copy()
wf["year"]=wf.evaluation_start.dt.year
family_labels={"sim_only":"Simulation-only neural family","market_only":"Market-only neural family","black_scholes_delta":"Black-Scholes delta","vendor_delta":"Vendor delta"}
fig,ax=plt.subplots(figsize=(9.4,5.4))
for fam,g in wf.groupby("policy_family"):
    ax.plot(g.year,g.premium_normalised_rmse,marker="o",label=family_labels[fam])
ax.set_xticks(sorted(wf.year.unique()))
ax.set_xlabel("Forward evaluation year"); ax.set_ylabel("Premium-normalised RMSE")
ax.set_title("Calendar variation in forward hedging performance")
ax.grid(alpha=.25); ax.legend(frameon=False,ncol=2)
save_show(fig,"12_walk_forward_family_rmse.png",
 title="Walk-forward performance by policy family",category="Compatible with final design",
 placement="Appendix temporal robustness. The paired annual-effect chart should remain the main-text figure.",
 importance="Shows that calendar variation is much larger than many average policy differences and identifies difficult years shared across policies.",
 caveats="Family averages can hide cell-level heterogeneity and are partly redundant with the paired annual-effect figure already in the dissertation.",
 recommendation="APPROVE - appendix only")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/12_walk_forward_family_rmse.png')

## 13. Concurrent liability and notional scale

In [14]:

fig,axes=plt.subplots(2,1,figsize=(10.5,6.4),sharex=True)
axes[0].plot(concurrent_load.date,concurrent_load.open_straddles)
axes[0].set_ylabel("Open straddles"); axes[0].set_title("Concurrent evaluated liabilities")
axes[1].plot(concurrent_load.date,concurrent_load.aggregate_index_notional_usd/1_000_000)
axes[1].set_ylabel("Index notional (USD millions)"); axes[1].set_xlabel("Date"); axes[1].set_title("Aggregate one-contract index notional")
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout()
save_show(fig,"13_concurrent_liability_capital_load.png",
 title="Concurrent liability and aggregate-notional load",category="Compatible with final design",
 placement="Appendix practical-scale diagnostics; cite in the external-validity discussion.",
 importance="Prevents one-contract episode results from being mistaken for a small retail account and quantifies the scale created by overlapping weekly entries.",
 caveats="Index notional is not broker margin, economic capital or funded exposure. Portfolio netting and collateral remuneration are absent.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/13_concurrent_liability_capital_load.png')

## 14. Cumulative paired dollar difference

In [15]:

fig,axes=plt.subplots(1,2,figsize=(12,4.8),sharey=True)
for ax,baseline in zip(axes,[BS,VENDOR]):
    g=cumulative_paths.loc[cumulative_paths.baseline_cell.eq(baseline)]
    for seed,sg in g.groupby("training_seed"):
        sg=sg.sort_values("exit_date")
        ax.plot(sg.exit_date,sg.cumulative_dollar_difference,alpha=.28,linewidth=1)
    med=g.groupby("exit_date",as_index=False).cumulative_dollar_difference.median().sort_values("exit_date")
    ax.plot(med.exit_date,med.cumulative_dollar_difference,linewidth=2.3,label="Median across seeds")
    ax.axhline(0,color="black",linewidth=1)
    ax.set_title(f"Relative to {LABELS[baseline]}"); ax.set_xlabel("Episode exit date"); ax.grid(alpha=.25)
axes[0].set_ylabel("Cumulative paired dollar P&L difference")
axes[1].legend(frameon=False)
fig.suptitle("Evaluated liability-book difference, not a funded equity curve")
fig.tight_layout()
save_show(fig,"14_cumulative_paired_dollar_difference.png",
 title="Cumulative paired dollar difference",category="Compatible with final design",
 placement="Appendix only, under an explicit heading that this is not a trading backtest.",
 importance="Shows when the dollar disadvantage accumulates and whether all fitted seeds tell the same temporal story.",
 caveats="Terminal P&L is booked at episode exit, liabilities overlap, and no margin or daily mark-to-market account is modelled. It must never be called an equity curve.",
 recommendation="CONDITIONAL - appendix only")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/14_cumulative_paired_dollar_difference.png')

## 15. Fold support and cohort exclusions

In [16]:

eval_support=fold_support.loc[(fold_support.set.eq("evaluation")) & (fold_support.source.eq("market"))].copy()
eval_support["year"]=eval_support.fold.str.extract(r"(20\d{2})").astype(float)
sup=eval_support.pivot(index="year",columns="bucket",values="episodes").reindex(columns=["1W","2W","1M","2M","3M"])
ex=cohort_exclusions.groupby(["reason","bucket"]).size().unstack("bucket",fill_value=0).reindex(columns=["1W","2W","1M","2M","3M"])
fig,axes=plt.subplots(1,2,figsize=(12.2,5.3))
for ax,frame,title,cmap in [(axes[0],sup,"Evaluation support by scheduled fold","YlGnBu"),(axes[1],ex,"Excluded cohorts by reason and maturity","Oranges")]:
    im=ax.imshow(frame.to_numpy(float),aspect="auto",cmap=cmap)
    ax.set_xticks(range(frame.shape[1])); ax.set_xticklabels(frame.columns)
    ax.set_yticks(range(frame.shape[0])); ax.set_yticklabels([str(int(x)) if isinstance(x,(float,np.floating)) and np.isfinite(x) else str(x).replace("_"," ") for x in frame.index])
    ax.set_title(title)
    for i in range(frame.shape[0]):
        for j in range(frame.shape[1]):
            v=frame.iloc[i,j]
            if pd.notna(v): ax.text(j,i,f"{v:.0f}",ha="center",va="center",fontsize=7)
fig.tight_layout()
save_show(fig,"15_fold_support_and_exclusions.png",
 title="Fold support and cohort-exclusion heatmaps",category="Compatible with final design",
 placement="Appendix data-validity section, referenced from the evidence-base paragraph.",
 importance="Separates genuine historical support limitations from discretionary filtering and documents why early folds are excluded before fitting.",
 caveats="Counts are diagnostic; they do not measure representativeness of the surviving episodes within each year.",
 recommendation="APPROVE - appendix")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/15_fold_support_and_exclusions.png')

## 16. Illustrative collateral sensitivity

In [17]:

models=[PRIMARY,MARKET,BS,VENDOR]
col=collateral.loc[collateral.model_cell.isin(models)].copy()
summary=col.groupby(["model_cell","starting_collateral_usd"]).agg(
    ret=("return_on_illustrative_collateral_pct","mean"),
    drawdown=("maximum_realised_drawdown_pct","mean"),
).reset_index()
fig,axes=plt.subplots(1,2,figsize=(11.3,4.7))
for model in models:
    g=summary.loc[summary.model_cell.eq(model)]
    axes[0].plot(g.starting_collateral_usd/1_000_000,g.ret,marker="o",label=LABELS[model])
    axes[1].plot(g.starting_collateral_usd/1_000_000,g.drawdown,marker="o",label=LABELS[model])
axes[0].set_ylabel("Illustrative realised return (%)"); axes[0].set_title("Terminal P&L / assumed collateral")
axes[1].set_ylabel("Illustrative maximum drawdown (%)"); axes[1].set_title("Realised drawdown / assumed collateral")
for ax in axes: ax.set_xlabel("Assumed starting collateral (USD millions)"); ax.grid(alpha=.25)
axes[1].legend(frameon=False,fontsize=7)
fig.suptitle("Scale illustration only: no broker-margin model")
fig.tight_layout()
save_show(fig,"16_collateral_sensitivity.png",
 title="One-contract collateral sensitivity",category="Compatible with final design",
 placement="Appendix economic-scale diagnostics, next to concurrent-notional load.",
 importance="Illustrates why raw dollar P&L cannot be interpreted without a capital denominator and shows how arbitrary collateral assumptions mechanically change percentage returns.",
 caveats="This is not a margin backtest. Collateral levels are illustrative; portfolio netting, variation margin, funding and regulatory capital are absent.",
 recommendation="CONDITIONAL - appendix only")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/16_collateral_sensitivity.png')

## 17. Representative-policy RMSE leaderboard

In [18]:

keys=[PRIMARY,"lifted_heston__pooled__head_only","lifted_heston__pooled__full","heston__partial__full",MARKET,VENDOR,BS,"heston_delta","lifted_heston_delta","no_hedge"]
rep=headline.loc[headline.underlying_cost_bps.eq(1.0)&headline.model_cell.isin(keys)].copy()
rep["label"]=rep.model_cell.map(LABELS).fillna(rep.model_cell)
rep=rep.sort_values("premium_normalised_rmse",ascending=False)
fig,ax=plt.subplots(figsize=(9.2,5.8))
ax.barh(rep.label,rep.premium_normalised_rmse)
ax.set_xlabel("Premium-normalised RMSE")
ax.set_title("Selected policy and benchmark point estimates at one basis point")
for i,v in enumerate(rep.premium_normalised_rmse): ax.text(v,i,f" {v:.3f}",va="center",fontsize=8)
ax.grid(axis="x",alpha=.25)
save_show(fig,"17_representative_policy_rmse.png",
 title="Representative-policy RMSE leaderboard",category="Compatible with final design",
 placement="Do not add to the main text; optional review appendix only.",
 importance="Provides a quick orientation to magnitudes across representative policies and analytical baselines.",
 caveats="A leaderboard encourages post-selection and obscures uncertainty, mechanisms, turnover and dollar outcomes. The existing headline table already carries this information more responsibly.",
 recommendation="REJECT from dissertation")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/17_representative_policy_rmse.png')

## 18. Upstream calibration-fit distribution

In [19]:

src=SUPERSEDED_DIR/"Part_C_Robustness_Checks_c11_o0.png"
dst=FIGURE_DIR/"18_upstream_calibration_fit_distribution.png"
shutil.copy2(src,dst)
REVIEW_ROWS.append({
    "title":"Upstream calibration-fit distribution",
    "category":"Upstream or superseded evidence",
    "filename":dst.name,
    "proposed_placement":"At most, a single upstream-calibration figure in an appendix; otherwise reference the calibration notebook.",
    "why_important":"Shows that pricing calibration quality and terminal control performance are separate empirical objects, and that the richer model did not automatically have lower median IV error in this earlier audit.",
    "caveats":"This chart comes from an earlier calibration specification and is not regenerated from the final H035 locked output. It must not be treated as a final-design result.",
    "preliminary_recommendation":"CONDITIONAL - upstream appendix only",
})
display(Image(filename=str(dst)))


<IPython.core.display.Image object>

# Superseded diagnostic gallery

The following items were reported in earlier branches of the project. They are included for approval or rejection, but **they are not compatible evidence for the final locked H035 observable-only MLP experiment** unless explicitly stated otherwise.

## S1. Earlier H025/H035 headline diagnostic suite

In [20]:

root=OLD_MEDIA_DIR/"Final_Dissertation_Observable_Heston_Lifted_H025_Audited_Expanded_With_Result_Placeholders"
paths=[root/f"image{i}.png" for i in range(1,6)]
thumbs=[]
for p in paths:
    im=PILImage.open(p).convert("RGB"); im.thumbnail((780,430)); thumbs.append((p.name,im.copy()))
canvas=PILImage.new("RGB",(1600,1400),"white"); draw=ImageDraw.Draw(canvas)
positions=[(10,10),(810,10),(10,470),(810,470),(410,930)]
for (name,im),(x,y) in zip(thumbs,positions):
    canvas.paste(im,(x+(780-im.width)//2,y)); draw.text((x+10,y+435),name,fill="black")
out=FIGURE_DIR/"S1_old_h025_diagnostic_suite.png"; canvas.save(out)
REVIEW_ROWS.append({"title":"Earlier H025/H035 headline diagnostic suite","category":"Superseded design","filename":out.name,
"proposed_placement":"None in the final results; retain in research-history archive.",
"why_important":"Shows the evolution of cost, maturity, held-out and exact-factor diagnostics and explains why the final project moved toward stricter timing and observable-only identification.",
"caveats":"Different timing, data splits, architectures and calibration specification. Numerical comparisons are not commensurate with the final H035 experiment.",
"preliminary_recommendation":"REJECT from final dissertation"})
display(Image(filename=str(out)))


<IPython.core.display.Image object>

## S2. Current-state MLP versus GRU and lag-stack MLP

In [21]:

labels=["Black-Scholes delta","Current-state MLP","GRU","Lag-stack MLP"]
values=[0.2951,0.3022,0.3133,0.3171]
fig,ax=plt.subplots(figsize=(8.7,4.5))
order=np.argsort(values)[::-1]
ax.barh(np.array(labels)[order],np.array(values)[order])
ax.set_xlabel("Premium-normalised RMSE")
ax.set_title("Superseded lag-consistent architecture comparison")
for i,v in enumerate(np.array(values)[order]): ax.text(v,i,f" {v:.4f}",va="center")
save_show(fig,"S2_old_architecture_comparison.png",
 title="Current-state MLP versus GRU and lag-stack MLP",category="Superseded design",
 placement="Future-research motivation only; do not merge with final results.",
 importance="Addresses the original question of whether explicit history or recurrence adds value beyond the current observable state.",
 caveats="Reported under an earlier lag-consistent design with only three seeds and a different experiment. It does not test the final 30-cell factorial specification.",
 recommendation="REJECT from final results; mention as exploratory history if useful")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/S2_old_architecture_comparison.png')

## S3. Short-end residual-memory stress diagnostics

In [22]:

src=Path("/mnt/data/diagnostic_review/part_E_contact.png")
dst=FIGURE_DIR/"S3_short_end_memory_stress.png"; shutil.copy2(src,dst)
REVIEW_ROWS.append({"title":"Short-end residual-memory stress diagnostics","category":"Superseded design","filename":dst.name,
"proposed_placement":"Future-research appendix only, if the dissertation discusses why recurrence was not retained.",
"why_important":"Explores whether memory-aware policies behave differently in short-dated and stressed scenarios, where rough-volatility and path-dependence arguments are strongest.",
"caveats":"Scenario labels, objectives and policy families differ from the final experiment; the dense panels are also difficult to communicate cleanly.",
"preliminary_recommendation":"REJECT from final results"})
display(Image(filename=str(dst)))


<IPython.core.display.Image object>

## S4. Lifted-factor mechanism and oracle-hedging suite

In [23]:

fig,ax=plt.subplots(figsize=(11,4.6)); ax.axis("off")
boxes=[
    (0.02,0.55,0.17,0.27,"Match states\nSame spot and variance\nDifferent factor mix"),
    (0.22,0.55,0.17,0.27,"Forecast future\nintegrated variance\nRaw vs projected factors"),
    (0.42,0.55,0.17,0.27,"Estimate oracle\nstock-hedge correction\nNested continuations"),
    (0.62,0.55,0.17,0.27,"Add variance instrument\nTest unspanned\nvolatility information"),
    (0.82,0.55,0.16,0.27,"Compare learned action\nwith oracle direction\nPolicy exploitation"),
]
for x,y,w,h,text in boxes:
    ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,linewidth=1.5)); ax.text(x+w/2,y+h/2,text,ha="center",va="center",fontsize=9)
for i in range(len(boxes)-1):
    x=boxes[i][0]+boxes[i][2]; nx=boxes[i+1][0]
    ax.annotate("",xy=(nx,0.685),xytext=(x,0.685),arrowprops=dict(arrowstyle="->",lw=1.3))
ax.text(.5,.25,"Claim ladder: predictive information  →  stock-hedgeable information  →  representation success  →  policy exploitation",ha="center",fontsize=11)
ax.set_title("Superseded lifted-factor mechanism programme")
save_show(fig,"S4_lifted_factor_mechanism_suite.png",
 title="Lifted-factor mechanism and oracle-hedging suite",category="Superseded design",
 placement="Future-research section as a compact proposed experiment, not an empirical result.",
 importance="This is the cleanest route to determining whether lifted-factor memory is predictive, spanned by the stock and actually used by a trained policy.",
 caveats="The accessible archive contains the designed tests but no completed final-output chart. The suite uses latent factors and an H030 branch, conflicting with the final observable-only H035 design.",
 recommendation="APPROVE as future-research schematic only")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/S4_lifted_factor_mechanism_suite.png')

## S5. JEPA market-state representation programme

In [24]:

fig,ax=plt.subplots(figsize=(10.5,4.4)); ax.axis("off")
labels=[("Observable market history",.05),("Context encoder",.27),("Predictive latent target",.49),("Hedging policy",.71),("Chronological confirmation",.90)]
for text,x in labels:
    ax.add_patch(plt.Rectangle((x-.085,.48,.17,.25),.17,.25,fill=False,linewidth=1.5))
    ax.text(x,.605,text,ha="center",va="center",fontsize=9,wrap=True)
for (t1,x1),(t2,x2) in zip(labels,labels[1:]): ax.annotate("",xy=(x2-.09,.605),xytext=(x1+.09,.605),arrowprops=dict(arrowstyle="->"))
ax.text(.5,.25,"Question: can self-supervised state prediction improve the representation used by the hedge without leaking future information?",ha="center",fontsize=10)
ax.set_title("Superseded JEPA representation-learning branch")
save_show(fig,"S5_jepa_representation_programme.png",
 title="JEPA market-state representation diagnostics",category="Superseded design",
 placement="Do not include in the final dissertation; retain as a separate future project.",
 importance="Represents a sophisticated alternative to hand-designed state variables by learning predictive market representations before the hedging objective.",
 caveats="No recoverable executed result chart was present in the accessible notebook. Including the method in the final dissertation would introduce an untested architecture outside the locked grid.",
 recommendation="REJECT from final dissertation")


<IPython.core.display.Image object>

PosixPath('/mnt/data/diagnostic_review/updated_figures/S5_jepa_representation_programme.png')

## S6. Earlier realised-SPX replay and strategy-style diagnostics

In [25]:

root=OLD_MEDIA_DIR/"Final_Dissertation_Observable_Heston_Lifted_H025_Audited_Expanded_With_Result_Placeholders"
paths=[root/"image1.png",root/"image2.png",root/"image3.png"]
canvas=PILImage.new("RGB",(1600,1050),"white"); draw=ImageDraw.Draw(canvas)
positions=[(10,10),(810,10),(410,540)]
for p,(x,y) in zip(paths,positions):
    im=PILImage.open(p).convert("RGB"); im.thumbnail((780,470)); canvas.paste(im,(x+(780-im.width)//2,y)); draw.text((x+10,y+475),p.name,fill="black")
out=FIGURE_DIR/"S6_old_strategy_replay.png"; canvas.save(out)
REVIEW_ROWS.append({"title":"Earlier realised-SPX replay and strategy-style diagnostics","category":"Superseded design","filename":out.name,
"proposed_placement":"None in the final empirical chapter; research-history archive only.",
"why_important":"Shows the earlier emphasis on held-out rankings, maturity performance and transaction-cost robustness that motivated the final walk-forward design.",
"caveats":"Earlier splits and execution conventions created look-ahead and comparability concerns. The charts should not be mixed with final estimates.",
"preliminary_recommendation":"REJECT from final dissertation"})
display(Image(filename=str(out)))


<IPython.core.display.Image object>

## S7. Conditional-likelihood versus Euler numerical comparison

In [26]:

paths=[SUPERSEDED_DIR/"CLP_Euler_c16_o0.png",SUPERSEDED_DIR/"CLP_Euler_c16_o1.png"]
canvas=PILImage.new("RGB",(1400,520),"white")
for p,x in zip(paths,[0,700]):
    im=PILImage.open(p).convert("RGB"); im.thumbnail((690,500)); canvas.paste(im,(x+(700-im.width)//2,(520-im.height)//2))
out=FIGURE_DIR/"S7_clp_vs_euler.png"; canvas.save(out)
REVIEW_ROWS.append({"title":"Conditional-likelihood versus Euler numerical comparison","category":"Superseded or upstream numerical evidence","filename":out.name,
"proposed_placement":"Optional numerical-method appendix only, if simulation discretisation receives explicit discussion.",
"why_important":"Demonstrates why a more stable simulation scheme was considered for lifted-volatility dynamics and guards against mistaking discretisation error for economic model behaviour.",
"caveats":"Teaching/numerical experiment rather than a direct validation of the final calibrated banks. It does not establish better market hedging performance.",
"preliminary_recommendation":"CONDITIONAL - numerical appendix only"})
display(Image(filename=str(out)))


<IPython.core.display.Image object>

# Execution audit

In [27]:

manifest=pd.DataFrame(REVIEW_ROWS)
manifest.insert(0,"item_id",[f"D{i:02d}" for i in range(1,len(manifest)+1)])
manifest["path"]=[str(FIGURE_DIR/f) for f in manifest.filename]
manifest["exists"]=[Path(p).is_file() for p in manifest.path]
manifest["bytes"]=[Path(p).stat().st_size if Path(p).is_file() else 0 for p in manifest.path]
manifest.to_csv(FIGURE_DIR/"diagnostic_review_manifest.csv",index=False)
(FIGURE_DIR/"diagnostic_review_manifest.json").write_text(manifest.to_json(orient="records",indent=2))
assert manifest.exists.all()
assert (manifest.bytes>10_000).all(), manifest.loc[manifest.bytes<=10_000,["title","bytes"]]
print({
    "status":"PASS",
    "figures":len(manifest),
    "compatible":int(manifest.category.eq("Compatible with final design").sum()),
    "upstream_or_superseded":int((~manifest.category.eq("Compatible with final design")).sum()),
    "training_performed":False,
    "manifest":str(FIGURE_DIR/"diagnostic_review_manifest.csv"),
})
display(manifest[["item_id","title","category","preliminary_recommendation"]])


{'status': 'PASS', 'figures': 25, 'compatible': 17, 'upstream_or_superseded': 8, 'training_performed': False, 'manifest': '/mnt/data/diagnostic_review/updated_figures/diagnostic_review_manifest.csv'}


,item_id,title,category,preliminary_recommendation
0,D01,Terminal-loss exceedance curves,Compatible with final design,APPROVE - main text
1,D02,Risk-turnover frontier,Compatible with final design,APPROVE - main text
2,D03,Paired one-contract dollar P&L difference,Compatible with final design,APPROVE - main text
3,D04,Causal regime-conditioned performance,Compatible with final design,APPROVE - appendix; cite in main discussion
4,D05,Accounting and P&L-component decomposition,Compatible with final design,APPROVE - appendix
5,D06,Optimisation-seed stability,Compatible with final design,APPROVE - appendix
6,D07,Transaction-cost sensitivity and tested break-...,Compatible with final design,APPROVE - appendix
7,D08,Paired win rates by maturity and cost,Compatible with final design,APPROVE - appendix
8,D09,Training and validation histories,Compatible with final design,APPROVE - appendix
9,D10,Quote funnel and maturity-by-year coverage,Compatible with final design,APPROVE - data chapter or Appendix A
